# Airline Loyalty — Churn & Retention Pipeline

Scores 13,191 active members (Jul 2017–Jun 2018) for disengagement risk in H2 2018.  
Outputs `final_customer_results.csv` which feeds the Streamlit dashboard directly.

**Files needed in the same directory:** `Customer_Loyalty_History.csv`, `Customer_Flight_Activity.csv`, `Calendar.csv`

In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_score,
                             recall_score, f1_score)
from xgboost import XGBClassifier

pd.set_option('display.width', 160)
RNG = 0

clh = pd.read_csv('Customer_Loyalty_History.csv')
cfa = pd.read_csv('Customer_Flight_Activity.csv')
cal = pd.read_csv('Calendar.csv', parse_dates=['Date', 'Start of Quarter'])

print(f'cfa: {len(cfa):,} rows | clh: {len(clh):,} rows')

# De-duplicate: drop exact copies first, then sum any remaining (year, month) conflicts
keys = ['Loyalty Number', 'Year', 'Month']
n_dup_keys = cfa.duplicated(subset=keys).sum()
n_dup_full  = cfa.duplicated().sum()
print(f'Duplicate keys: {n_dup_keys:,}  (exact: {n_dup_full:,}, conflicting: {n_dup_keys - n_dup_full:,})')

cfa_cleaned = (cfa.drop_duplicates()
                  .groupby(keys, as_index=False)
                  .sum())
print(f'After dedup: {len(cfa_cleaned):,} rows')

clh_cleaned = clh.copy()
n_neg = (clh_cleaned['Salary'] < 0).sum()
clh_cleaned['Salary'] = clh_cleaned['Salary'].abs()
clh_cleaned = clh_cleaned.drop(columns=['Country'])   # constant field across all rows
print(f'Negative salaries corrected: {n_neg}')

cfa_cleaned['ym'] = cfa_cleaned['Year'] * 12 + cfa_cleaned['Month']
clh_cleaned['cancel_ym'] = (clh_cleaned['Cancellation Year'] * 12
                             + clh_cleaned['Cancellation Month'])

cfa: 392,936 rows | clh: 16,737 rows
Duplicate keys: 3,871  (exact: 1,922, conflicting: 1,949)
After dedup: 389,065 rows
Negative salaries corrected: 20


In [20]:
# Prediction point T = end of June 2018.
# Population: flew at least once Jul 2017–Jun 2018 AND not cancelled by June 2018.
# Churn: no flights in H2 2018 OR membership cancelled in H2 2018.
T        = 2018 * 12 + 6
END_2018 = 2018 * 12 + 12

recent12       = cfa_cleaned[(cfa_cleaned['ym'] >= T - 11) & (cfa_cleaned['ym'] <= T)]
recent_flights = recent12.groupby('Loyalty Number')['Total Flights'].sum()
active_recent  = recent_flights[recent_flights > 0]
cancelled_by_T = set(clh_cleaned.loc[clh_cleaned['cancel_ym'] <= T, 'Loyalty Number'])
population_ids = set(active_recent.index) - cancelled_by_T

h2           = cfa_cleaned[(cfa_cleaned['ym'] > T) & (cfa_cleaned['ym'] <= END_2018)]
h2_flights   = h2.groupby('Loyalty Number')['Total Flights'].sum()
flew_h2      = set(h2_flights[h2_flights > 0].index)
cancelled_h2 = set(clh_cleaned.loc[(clh_cleaned['cancel_ym'] > T) &
                                    (clh_cleaned['cancel_ym'] <= END_2018), 'Loyalty Number'])
churned_ids  = (population_ids - flew_h2) | (population_ids & cancelled_h2)

customers = pd.DataFrame(index=pd.Index(sorted(population_ids), name='Loyalty Number'))
customers['churned'] = customers.index.isin(churned_ids).astype(int)

print(f'Population: {len(customers):,} members | '
      f'Churners: {customers["churned"].sum():,} ({customers["churned"].mean()*100:.1f}%)')

Population: 13,191 members | Churners: 422 (3.2%)


In [21]:
hist   = cfa_cleaned[cfa_cleaned['ym'] <= T]
active = hist[hist['Total Flights'] > 0]

def window(df, n):
    return df[(df['ym'] >= T - n + 1) & (df['ym'] <= T)]

def agg(df, col, how='sum'):
    return getattr(df.groupby('Loyalty Number')[col], how)().reindex(customers.index).fillna(0)

w3, w6, w12 = window(hist, 3), window(hist, 6), window(hist, 12)

# Recency
last_active = active.groupby('Loyalty Number')['ym'].max()
customers['recency_months'] = (T - last_active).reindex(customers.index).fillna(99)

# Frequency at 3m / 6m / 12m horizons
customers['flights_3m']  = agg(w3,  'Total Flights')
customers['flights_6m']  = agg(w6,  'Total Flights')
customers['flights_12m'] = agg(w12, 'Total Flights')

act3  = w3[w3['Total Flights'] > 0]
act6  = w6[w6['Total Flights'] > 0]
act12 = w12[w12['Total Flights'] > 0]
customers['active_months_3m']  = agg(act3,  'Total Flights', 'count')
customers['active_months_6m']  = agg(act6,  'Total Flights', 'count')
customers['active_months_12m'] = agg(act12, 'Total Flights', 'count')

# Momentum: recent vs prior 6m
prior6 = hist[(hist['ym'] >= T - 11) & (hist['ym'] <= T - 6)]
customers['flights_prior6']    = agg(prior6, 'Total Flights')
customers['trend_6v6']         = customers['flights_6m'] - customers['flights_prior6']
customers['ratio_6v6']         = (customers['flights_6m'] + 1) / (customers['flights_prior6'] + 1)
# 0 = activity front-loaded (fading member); 1 = all activity in last 3 months
customers['share_last3_of_12'] = customers['flights_3m'] / (customers['flights_12m'] + 1)

# Recency-weighted activity: recent months count more (0.8^k decay)
w = w12.copy()
w['wgt']      = 0.8 ** (T - w['ym'])
w['wflights'] = w['wgt'] * w['Total Flights']
customers['flights_decayed'] = agg(w, 'wflights')

# Distance & intensity
customers['distance_12m']             = agg(w12, 'Distance')
customers['distance_per_flight']      = customers['distance_12m'] / (customers['flights_12m'] + 1)
customers['avg_flights_per_active_month'] = (
    customers['flights_12m'] / customers['active_months_12m'].replace(0, np.nan)).fillna(0)

# Points engagement
customers['points_acc_12m']    = agg(w12, 'Points Accumulated')
customers['points_red_12m']    = agg(w12, 'Points Redeemed')
customers['redemption_ratio']  = customers['points_red_12m'] / (customers['points_acc_12m'] + 1)
customers['ever_redeemed_12m'] = (customers['points_red_12m'] > 0).astype(int)
red6 = agg(w6, 'Points Redeemed')
customers['redeemed_6m_flag']  = (red6 > 0).astype(int)

# Regularity: longest inactive gap and current streak
piv = w12.pivot_table(index='Loyalty Number', columns='ym',
                       values='Total Flights', aggfunc='sum', fill_value=0)
piv = (piv.reindex(columns=range(T - 11, T + 1), fill_value=0)
          .reindex(customers.index, fill_value=0))

def longest_zero_run_and_tail(row):
    longest = run = tail = 0
    for v in row:
        run = run + 1 if v == 0 else 0
        longest = max(longest, run)
    for v in row[::-1]:
        if v == 0: tail += 1
        else: break
    return longest, tail

zr = np.array([longest_zero_run_and_tail(r) for r in piv.values])
customers['max_zero_run_12m']  = zr[:, 0]
customers['zero_streak_now']   = zr[:, 1]
customers['flight_std_12m']    = piv.std(axis=1)
customers['flight_cv_12m']     = piv.std(axis=1) / (piv.mean(axis=1) + 1)
top3 = np.sort(piv.values, axis=1)[:, -3:].sum(axis=1)
customers['top3_month_share']  = top3 / (piv.sum(axis=1).values + 1)

# Distance and points deceleration
customers['flights_1m']   = agg(window(hist, 1), 'Total Flights')
d6 = agg(w6, 'Distance')
customers['dist_ratio_6v6'] = (d6 + 1) / ((customers['distance_12m'] - d6) + 1)
p6 = agg(w6, 'Points Accumulated')
customers['pts_ratio_6v6']  = (p6 + 1) / ((customers['points_acc_12m'] - p6) + 1)

# Seasonal features (Calendar.csv provides quarter mapping)
# Peak travel months in Canada: summer Jun–Aug, December holiday
cal['Year']    = cal['Date'].dt.year
cal['Month']   = cal['Date'].dt.month
cal['quarter'] = ((cal['Start of Quarter'].dt.month - 1) // 3) + 1
month_q = cal.drop_duplicates(subset=['Year', 'Month'])[['Year', 'Month', 'quarter']]

PEAK_MONTHS = {6, 7, 8, 12}
w12_peak = w12[w12['Month'].isin(PEAK_MONTHS)]
customers['peak_flights_12m']  = agg(w12_peak, 'Total Flights')
customers['peak_season_share'] = customers['peak_flights_12m'] / (customers['flights_12m'] + 1)

w12_q = w12.merge(month_q, on=['Year', 'Month'], how='left')
q_piv = (w12_q.groupby(['Loyalty Number', 'quarter'])['Total Flights']
               .sum().unstack(fill_value=0)
               .reindex(customers.index, fill_value=0))
customers['quarter_concentration'] = q_piv.max(axis=1) / (q_piv.sum(axis=1) + 1)

# Tenure
clh_idx   = clh_cleaned.set_index('Loyalty Number')
enroll_ym = clh_idx['Enrollment Year'] * 12 + clh_idx['Enrollment Month']
customers['tenure_months'] = (T - enroll_ym).reindex(customers.index)

# Demographics (one-hot; CLV and Salary excluded to prevent leakage)
demo    = clh_idx.reindex(customers.index)[['Loyalty Card', 'Education',
                                             'Marital Status', 'Gender', 'Enrollment Type']]
demo_oh = pd.get_dummies(demo, drop_first=True, dtype=int)
customers = customers.join(demo_oh)

print(f'Feature matrix: {customers.shape[0]:,} × {customers.shape[1]-1} features')
assert customers.drop(columns='churned').isnull().sum().sum() == 0, 'NaN check failed'
print('NaN check passed')

Feature matrix: 13,191 × 42 features
NaN check passed


In [22]:
y = customers['churned']
X = customers.drop(columns=['churned'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG)
print(f'Train: {len(X_train):,} ({y_train.sum()} churners) | '
      f'Test:  {len(X_test):,} ({y_test.sum()} churners)')

spw = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=600, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=8,
    gamma=1, reg_lambda=2, scale_pos_weight=spw,
    eval_metric='aucpr', random_state=RNG, n_jobs=-1)

# 5-fold CV on the training set. OOF probabilities are used for threshold selection
# so no test-set information leaks into the threshold choice.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
oof_proba = cross_val_predict(model, X_train, y_train, cv=cv,
                               method='predict_proba', n_jobs=-1)[:, 1]
cv_auc = roc_auc_score(y_train, oof_proba)
cv_pr  = average_precision_score(y_train, oof_proba)
print(f'\n5-fold CV (OOF): ROC AUC = {cv_auc:.3f} | PR AUC = {cv_pr:.3f}  '
      f'(no-skill baseline = {y_train.mean():.3f})')

prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
f2_scores = 5 * prec * rec / (4 * prec + rec + 1e-9)   # recall weighted 2× precision
t_f1 = thr[np.argmax(f1_scores[:-1])]
t_f2 = thr[np.argmax(f2_scores[:-1])]

print('\nThreshold sweep (OOF):')
print(f'{"thr":>6} {"precision":>10} {"recall":>8} {"F1":>7} {"flagged":>9}')
for t in [0.30, 0.40, 0.50, 0.60, t_f1, t_f2, 0.70, 0.80]:
    p = (oof_proba >= t).astype(int)
    print(f'{t:6.2f} {precision_score(y_train, p, zero_division=0):10.3f} '
          f'{recall_score(y_train, p):8.3f} '
          f'{f1_score(y_train, p, zero_division=0):7.3f} {p.sum():9,}')
print(f'\nbest-F1 threshold: {t_f1:.3f} | best-F2 (recall-weighted): {t_f2:.3f}')

model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

print(f'\nHeld-out test set:')
print(f'ROC AUC = {roc_auc_score(y_test, proba_test):.3f} | '
      f'PR AUC = {average_precision_score(y_test, proba_test):.3f} '
      f'(baseline = {y_test.mean():.3f})')

for name, t in [('default 0.50', 0.50), (f'best-F1 {t_f1:.2f}', t_f1),
                (f'best-F2 {t_f2:.2f}', t_f2)]:
    preds = (proba_test >= t).astype(int)
    print(f'\nthreshold {name}:')
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, digits=3,
                                target_names=['retained', 'churned']))

dec = pd.DataFrame({'proba': proba_test, 'churned': y_test.values})
dec['decile'] = pd.qcut(dec['proba'].rank(method='first'), 10,
                        labels=range(10, 0, -1)).astype(int)
tab = dec.groupby('decile').agg(n=('churned', 'size'), churners=('churned', 'sum'))
tab = tab.sort_index()
tab['churn_rate']  = (tab['churners'] / tab['n']).round(3)
tab['cum_capture'] = (tab['churners'].cumsum() / dec['churned'].sum()).round(3)
print('\nCapture by decile (test set):')
print(tab.to_string())
print(f'\nTop 10% riskiest: {tab.iloc[0]["cum_capture"]*100:.0f}% of all churners captured; '
      f'top 20%: {tab.iloc[:2]["churners"].sum()/dec["churned"].sum()*100:.0f}%.')

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\nTop 15 features:')
print(imp.head(15).round(4).to_string())

Train: 9,893 (316 churners) | Test:  3,298 (106 churners)

5-fold CV (OOF): ROC AUC = 0.683 | PR AUC = 0.098  (no-skill baseline = 0.032)

Threshold sweep (OOF):
   thr  precision   recall      F1   flagged
  0.30      0.060    0.557   0.108     2,956
  0.40      0.080    0.462   0.137     1,814
  0.50      0.103    0.396   0.164     1,212
  0.60      0.126    0.342   0.184       855
  0.65      0.139    0.316   0.193       719
  0.57      0.122    0.370   0.183       961
  0.70      0.140    0.247   0.179       556
  0.80      0.141    0.111   0.124       248

best-F1 threshold: 0.649 | best-F2 (recall-weighted): 0.566

Held-out test set:
ROC AUC = 0.759 | PR AUC = 0.163 (baseline = 0.032)

threshold default 0.50:
[[2771  421]
 [  46   60]]
              precision    recall  f1-score   support

    retained      0.984     0.868     0.922      3192
     churned      0.125     0.566     0.204       106

    accuracy                          0.858      3298
   macro avg      0.554     0.

In [23]:
# Refit on all labeled data; 5-seed ensemble reduces run-to-run variance
seed_probas = []
for s in range(5):
    params = model.get_params()
    params['random_state'] = s
    fm = XGBClassifier(**params)
    fm.fit(X, y)
    seed_probas.append(fm.predict_proba(X)[:, 1])
customers['churn_probability'] = np.mean(seed_probas, axis=0)

# Risk bands: High = above best-F1 threshold; Medium between 60% of F2 and F1 threshold
def risk_band(p):
    if p >= t_f1:        return 'High'
    if p >= t_f2 * 0.6: return 'Medium'
    return 'Low'
customers['risk_band']    = customers['churn_probability'].apply(risk_band)
customers['CLV']          = clh_idx['CLV'].reindex(customers.index)
customers['loyalty_card'] = clh_idx['Loyalty Card'].reindex(customers.index)

clv_thresh = customers['CLV'].quantile(0.70)   # top 30% by lifetime value = "high value"

def segment(row):
    hv = row['CLV'] >= clv_thresh
    if row['risk_band'] == 'High' and hv:   return 'VIP at risk'
    if row['risk_band'] == 'High':          return 'At risk'
    if row['risk_band'] == 'Medium' and hv: return 'High-value, cooling'
    if row['risk_band'] == 'Medium':        return 'Cooling'
    if hv:                                  return 'High-value, healthy'
    return 'Healthy'
customers['segment'] = customers.apply(segment, axis=1)

ACTIONS = {
    'VIP at risk':         'Week 1: personal call from loyalty desk + targeted offer '
                           '(companion voucher or status extension). Owner: retention team.',
    'At risk':             'Automated win-back email within 7 days: bonus-points offer '
                           'valid 60 days, route suggestions from past trips.',
    'High-value, cooling': 'Monthly: lounge pass or seat-upgrade voucher to re-engage '
                           'before recency grows. Owner: CRM automation.',
    'Cooling':             'Quarterly re-engagement email + reminder of unspent points.',
    'High-value, healthy': 'Do not discount. Surprise-and-delight touchpoint 1x/quarter.',
    'Healthy':             'Standard newsletter cadence. No spend.',
}
customers['recommended_action'] = customers['segment'].map(ACTIONS)

print(customers.groupby('segment').agg(
    n=('churned', 'size'),
    actual_churn_rate=('churned', 'mean'),
    avg_probability=('churn_probability', 'mean'),
    avg_CLV=('CLV', 'mean')).round(3).to_string())

out = customers.reset_index()
out.to_csv('final_customer_results.csv', index=False)
print(f'\nSaved {len(out):,} customers → final_customer_results.csv')

                        n  actual_churn_rate  avg_probability    avg_CLV
segment                                                                 
At risk               826              0.228            0.787   4919.937
Cooling              1936              0.054            0.443   4845.887
Healthy              6470              0.000            0.185   4826.716
High-value, cooling   825              0.046            0.439  15466.293
High-value, healthy  2798              0.001            0.185  15145.134
VIP at risk           336              0.262            0.792  15340.222

Saved 13,191 customers → final_customer_results.csv


In [ ]:
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Retention Control Tower",
    page_icon="✈",
    layout="wide",
    initial_sidebar_state="collapsed",
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

html, body, [class*="css"] { font-family: 'Inter', sans-serif !important; }
.block-container { padding: 1rem 0 4rem 0 !important; max-width: 100% !important; }
footer, #MainMenu, header { display: none !important; }

.rct-header {
  background: #0f1f3d;
  color: white;
  padding: 20px 36px;
  display: flex;
  align-items: center;
  gap: 16px;
  margin-bottom: 24px;
}
.rct-header h1 { font-size: 20px; font-weight: 700; margin: 0; color: white !important; }
.rct-header p  { font-size: 12px; color: rgba(255,255,255,0.5); margin: 3px 0 0; }

.kpi-grid {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 16px;
  padding: 0 36px;
  margin-bottom: 28px;
}
.kpi-card {
  background: #ffffff;
  border: 1px solid rgba(15,31,61,0.12);
  border-top: 3px solid #888;
  border-radius: 10px;
  padding: 16px 18px;
}
.kpi-card.blue   { border-top-color: #2563eb; }
.kpi-card.red    { border-top-color: #dc2626; }
.kpi-card.amber  { border-top-color: #d97706; }
.kpi-card.purple { border-top-color: #7c3aed; }
.kpi-label { font-size: 11px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.6px; color: #64748b; }
.kpi-value { font-size: 28px; font-weight: 700; letter-spacing: -0.5px; color: #0f1f3d; margin: 6px 0 3px; line-height: 1; }
.kpi-sub   { font-size: 11px; color: #94a3b8; }

.seg-card {
  background: #ffffff;
  border: 1px solid rgba(15,31,61,0.1);
  border-left: 4px solid #888;
  border-radius: 8px;
  padding: 14px 18px;
  margin-bottom: 12px;
}
.seg-card h4   { font-size: 14px; font-weight: 600; color: #0f1f3d; margin: 0 0 3px; }
.seg-card .meta{ font-size: 12px; color: #64748b; margin-bottom: 6px; }
.seg-card .play{ font-size: 13px; color: #0f1f3d; }
.seg-card.red   { border-left-color: #dc2626; }
.seg-card.amber { border-left-color: #d97706; }
.seg-card.green { border-left-color: #16a34a; }

.summary-bar {
  background: #f1f5f9;
  border-radius: 8px;
  padding: 10px 16px;
  font-size: 13px;
  color: #64748b;
  margin-bottom: 16px;
}
.summary-bar strong { color: #0f1f3d; }

.stTabs [data-baseweb="tab-list"] {
  gap: 0 !important;
  border-bottom: 1px solid rgba(15,31,61,0.12) !important;
  background: transparent !important;
  padding: 0 36px !important;
}
.stTabs [data-baseweb="tab"] {
  padding: 10px 20px !important;
  font-size: 13px !important;
  font-weight: 500 !important;
  color: #64748b !important;
  background: transparent !important;
  border: none !important;
  border-bottom: 2px solid transparent !important;
}
.stTabs [aria-selected="true"] {
  color: #2563eb !important;
  border-bottom-color: #2563eb !important;
  background: transparent !important;
}
.stTabs [data-baseweb="tab-panel"] { padding: 24px 36px 0 36px !important; }
</style>
""", unsafe_allow_html=True)


# ── Data ──────────────────────────────────────────────────────────────────────

@st.cache_data
def load_data():
    df = pd.read_csv("final_customer_results.csv")
    df["Loyalty Number"] = df["Loyalty Number"].astype(int)
    return df

try:
    df = load_data()
except FileNotFoundError:
    st.error("**File not found:** `final_customer_results.csv` — run `Final_pipeline.ipynb` first.")
    st.stop()

SEG_ORDER = ["VIP at risk", "At risk", "High-value, cooling", "Cooling", "High-value, healthy", "Healthy"]
SEG_CSS   = {"VIP at risk": "red", "At risk": "red",
             "High-value, cooling": "amber", "Cooling": "amber",
             "High-value, healthy": "green", "Healthy": "green"}
RISK_HEX  = {"High": "#dc2626", "Medium": "#d97706", "Low": "#16a34a"}

at_risk     = df[df["risk_band"] == "High"]
vip_risk    = df[df["segment"] == "VIP at risk"]
clv_exposed = at_risk["CLV"].sum()


# ── Header ────────────────────────────────────────────────────────────────────

st.markdown(f"""
<div class="rct-header">
  <span style="font-size:30px">✈</span>
  <div>
    <h1>Retention Control Tower</h1>
    <p>Scored end of June 2018 · disengagement risk forecast for July–December 2018 · refresh by re-running Final_pipeline.ipynb</p>
  </div>
</div>
""", unsafe_allow_html=True)


# ── KPI strip ─────────────────────────────────────────────────────────────────

st.markdown(f"""
<div class="kpi-grid">
  <div class="kpi-card blue">
    <div class="kpi-label">Active members</div>
    <div class="kpi-value">{len(df):,}</div>
    <div class="kpi-sub">in scoring population</div>
  </div>
  <div class="kpi-card red">
    <div class="kpi-label">Flagged high risk</div>
    <div class="kpi-value">{len(at_risk):,}</div>
    <div class="kpi-sub">~1 in 5 disengage within 6 months</div>
  </div>
  <div class="kpi-card amber">
    <div class="kpi-label">VIPs at risk</div>
    <div class="kpi-value">{len(vip_risk):,}</div>
    <div class="kpi-sub">top-30% CLV — prioritise these first</div>
  </div>
  <div class="kpi-card purple">
    <div class="kpi-label">CLV exposed</div>
    <div class="kpi-value">${clv_exposed/1e6:,.1f}M</div>
    <div class="kpi-sub">lifetime value of all high-risk members</div>
  </div>
</div>
""", unsafe_allow_html=True)


# ── Tabs ──────────────────────────────────────────────────────────────────────

tab_list, tab_play, tab_member = st.tabs(
    ["📋  Action list", "🗺️  Segment playbook", "🔍  Member lookup"]
)

# ── Tab 1: Action list ────────────────────────────────────────────────────────

with tab_list:
    st.markdown("#### Who needs attention first")
    st.caption("Sorted by risk score. Filter, review, and download — ready to hand to the ops team.")

    f1, f2, f3 = st.columns([2, 2, 3])
    seg_pick  = f1.multiselect("Segments", SEG_ORDER, default=["VIP at risk", "At risk"])
    card_pick = f2.multiselect("Loyalty card", sorted(df["loyalty_card"].unique()),
                               default=sorted(df["loyalty_card"].unique()))
    top_n     = f3.slider("Members the team can contact", 50, 3000, 500, step=50)

    view = (df[df["segment"].isin(seg_pick) & df["loyalty_card"].isin(card_pick)]
              .sort_values("churn_probability", ascending=False)
              .head(top_n))

    if len(view) == 0:
        st.info("No members match these filters — try widening the segment or card selection.")
    else:
        exp_churn = view["churn_probability"].sum()
        st.markdown(
            f'<div class="summary-bar">Members selected: <strong>{len(view):,}</strong> &nbsp;·&nbsp; '
            f'Expected to disengage: <strong>{exp_churn:,.0f}</strong> &nbsp;·&nbsp; '
            f'Combined CLV: <strong>${view["CLV"].sum():,.0f}</strong></div>',
            unsafe_allow_html=True,
        )

        scatter_colors = {
            "VIP at risk": "#dc2626", "At risk": "#ef4444",
            "High-value, cooling": "#f59e0b", "Cooling": "#fbbf24",
            "High-value, healthy": "#16a34a", "Healthy": "#4ade80",
        }
        fig = px.scatter(
            view, x="churn_probability", y="CLV", color="segment",
            color_discrete_map=scatter_colors,
            hover_data=["Loyalty Number", "loyalty_card", "recency_months"],
            labels={"churn_probability": "Churn probability", "CLV": "Lifetime value ($)", "segment": "Segment"},
            height=300,
        )
        fig.update_traces(marker=dict(size=6, opacity=0.7))
        fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="#f8fafc",
            font_family="Inter, sans-serif", font_color="#0f1f3d",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
            margin=dict(t=40, b=36, l=0, r=0),
            xaxis=dict(gridcolor="rgba(0,0,0,0.05)", tickformat=".0%"),
            yaxis=dict(gridcolor="rgba(0,0,0,0.05)", tickprefix="$", tickformat=",.0f"),
        )
        st.plotly_chart(fig, use_container_width=True)

        show = view[[
            "Loyalty Number", "segment", "churn_probability", "CLV",
            "loyalty_card", "recency_months", "flights_12m",
            "zero_streak_now", "recommended_action",
        ]].rename(columns={
            "churn_probability": "Risk score", "segment": "Segment",
            "loyalty_card": "Card", "recency_months": "Months since last flight",
            "flights_12m": "Flights (12m)", "zero_streak_now": "Inactive streak (mo)",
            "recommended_action": "Next action",
        })

        st.dataframe(
            show, use_container_width=True, hide_index=True, height=360,
            column_config={
                "Risk score": st.column_config.ProgressColumn(
                    "Risk score", min_value=0, max_value=1, format="%.2f"),
                "CLV": st.column_config.NumberColumn(format="$%d"),
            },
        )
        st.download_button(
            "⬇  Download contact list (CSV)",
            show.to_csv(index=False).encode(),
            file_name="retention_contact_list.csv", mime="text/csv",
        )


# ── Tab 2: Segment playbook ───────────────────────────────────────────────────

with tab_play:
    st.markdown("#### One play per segment")
    st.caption("Six segments from two questions: how likely to disengage (model risk) and how valuable (top-30% CLV = high).")

    stats = df.groupby("segment").agg(
        n=("Loyalty Number", "size"),
        churn_rate=("churned", "mean"),
        clv=("CLV", "mean"),
        action=("recommended_action", "first"),
    )

    left, right = st.columns(2)
    for i, seg in enumerate(SEG_ORDER):
        if seg not in stats.index:
            continue
        r   = stats.loc[seg]
        css = SEG_CSS.get(seg, "green")
        html = f"""
        <div class="seg-card {css}">
          <h4>{seg}</h4>
          <div class="meta">{int(r['n']):,} members · {r['churn_rate']*100:.0f}% observed disengagement · avg CLV ${r['clv']:,.0f}</div>
          <div class="play"><strong>Play:</strong> {r['action']}</div>
        </div>"""
        (left if i % 2 == 0 else right).markdown(html, unsafe_allow_html=True)

    st.markdown("#### Where the value sits")
    st.caption("Total lifetime value ($M) by risk band — high-value bars in High/Medium = immediate revenue at stake.")

    pivot = df.pivot_table(
        index="risk_band",
        columns=df["CLV"] >= df["CLV"].quantile(0.70),
        values="CLV", aggfunc="sum",
    ).rename(columns={False: "Standard value", True: "High value (top 30%)"})
    pivot = pivot.reindex(["High", "Medium", "Low"]) / 1e6

    fig2 = go.Figure()
    for col, color in [("Standard value", "#93c5fd"), ("High value (top 30%)", "#1d4ed8")]:
        if col in pivot.columns:
            fig2.add_trace(go.Bar(name=col, x=pivot.index, y=pivot[col].round(2), marker_color=color))
    fig2.update_layout(
        barmode="stack",
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="#f8fafc",
        font_family="Inter, sans-serif", font_color="#0f1f3d",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
        margin=dict(t=40, b=36, l=0, r=0), height=260,
        xaxis=dict(gridcolor="rgba(0,0,0,0.05)"),
        yaxis=dict(gridcolor="rgba(0,0,0,0.05)", tickprefix="$", ticksuffix="M"),
    )
    st.plotly_chart(fig2, use_container_width=True)


# ── Tab 3: Member lookup ──────────────────────────────────────────────────────

with tab_member:
    st.markdown("#### Look up one member")
    q = st.text_input("Loyalty number", placeholder="e.g. 100018")

    if q:
        try:
            row = df[df["Loyalty Number"] == int(q)]
        except ValueError:
            row = df.iloc[0:0]

        if len(row) == 0:
            st.warning("Member not found. Members with no flights in the last 12 months are outside the scored population.")
        else:
            r         = row.iloc[0]
            risk_hex  = RISK_HEX.get(r.get("risk_band", "Low"), "#16a34a")

            st.markdown(f"""
            <div style="background:#fff;border:1px solid rgba(15,31,61,0.1);border-left:4px solid {risk_hex};
                        border-radius:10px;padding:18px 22px;margin-bottom:16px">
              <div style="font-size:12px;color:#64748b;margin-bottom:4px">Loyalty #{int(r['Loyalty Number'])}</div>
              <div style="font-size:22px;font-weight:700;color:#0f1f3d;margin-bottom:2px">{r['segment']}</div>
              <div style="font-size:13px;color:#64748b">Card: {r['loyalty_card']} &nbsp;·&nbsp; Risk band: {r.get('risk_band','—')}</div>
            </div>""", unsafe_allow_html=True)

            def mini(col, label, value):
                col.markdown(f"""
                <div style="background:#f8fafc;border-radius:8px;padding:12px 14px;margin-bottom:12px">
                  <div style="font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;color:#64748b">{label}</div>
                  <div style="font-size:22px;font-weight:700;color:#0f1f3d;margin:4px 0 0">{value}</div>
                </div>""", unsafe_allow_html=True)

            c1, c2, c3, c4 = st.columns(4)
            mini(c1, "Risk score",          f"{r['churn_probability']:.2f}")
            mini(c2, "Lifetime value",       f"${r['CLV']:,.0f}")
            mini(c3, "Flights (12m)",        int(r["flights_12m"]))
            mini(c4, "Tenure",               f"{int(r['tenure_months'])} mo")

            c5, c6 = st.columns(2)
            mini(c5, "Months since last flight", int(r["recency_months"]))
            mini(c6, "Inactive streak",          f"{int(r['zero_streak_now'])} mo")

            st.markdown(f"""
            <div style="background:#eff6ff;border-radius:8px;padding:14px 18px;font-size:13px;color:#1e3a8a">
              <strong>Recommended next action:</strong> {r['recommended_action']}
            </div>""", unsafe_allow_html=True)

2026-06-13 18:15:10.723 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 18:15:10.723 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 18:15:10.791 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-06-13 18:15:10.792 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 18:15:10.792 No runtime found, using MemoryCacheStorageManager
2026-06-13 18:15:10.792 No runtime found, using MemoryCacheStorageManager
2026-06-13 18:15:10.793 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 18:15:10.793 Thread 'MainThread': missing ScriptRunContext! This warning c

Exception ignored in: <function ResourceTracker.__del__ at 0x104115bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x105165bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x111865bc0>
Traceback (most recent call last